# 03 — Scaffold a New Agent

This notebook walks you through creating a new agent using the scaffolding script,
verifying the generated files, running the generated tests, deploying it locally,
and optionally deleting it.

## Prerequisites

- Python 3.11+
- Dependencies installed (`uv sync --group dev`)

## Configuration

Set the name for your new agent below. Must be kebab-case (e.g., `my-agent`).

In [5]:
import sys, pathlib

# Ensure the local project root is first on sys.path so our `agents`
# package takes priority over the openai-agents site-package.
_project_root = str(pathlib.Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# Change this to your desired agent name (kebab-case)
AGENT_NAME = "probation-q-a-agent"
MODEL = "gpt-5.1"

## Step 1: Scaffold the Agent

Run the scaffolding script with `--name` to generate all agent files, test stubs, and registry entry.

In [6]:
!python ../scripts/create_agent.py --name {AGENT_NAME} --model {MODEL}

[scaffold] ✗ Agent 'probation-q-a-agent' already exists at agents/probation_q_a_agent


## Step 2: Verify Generated Files

Check that all expected files were created in the agent and test directories.

In [7]:
import os

module_name = AGENT_NAME.replace("-", "_")
agent_dir = f"../agents/{module_name}"
test_dir = f"../tests/{module_name}"

print(f"Agent directory: agents/{module_name}/")
for root, dirs, files in os.walk(agent_dir):
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        print(f"{sub_indent}{f}")

print(f"\nTest directory: tests/{module_name}/")
for root, dirs, files in os.walk(test_dir):
    level = root.replace(test_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        print(f"{sub_indent}{f}")

Agent directory: agents/probation_q_a_agent/
probation_q_a_agent/
  README.md
  __init__.py
  config.py
  instructions.md
  __pycache__/
    __init__.cpython-312.pyc
    config.cpython-312.pyc
  integrations/
    __init__.py
    knowledge.py
  tools/
    __init__.py
    sample_tool.py
    __pycache__/
      __init__.cpython-312.pyc
      sample_tool.cpython-312.pyc

Test directory: tests/probation_q_a_agent/
probation_q_a_agent/
  __init__.py
  conftest.py
  test_agent_create.py
  test_agent_run.py
  test_tools.py
  __pycache__/
    __init__.cpython-312.pyc
    conftest.cpython-312-pytest-9.0.2.pyc
    test_agent_create.cpython-312-pytest-9.0.2.pyc
    test_agent_run.cpython-312-pytest-9.0.2.pyc
    test_tools.cpython-312-pytest-9.0.2.pyc


## Step 3: Inspect Generated Config

View the generated configuration to confirm the agent name, model, and paths.

In [8]:
config_path = f"../agents/{module_name}/config.py"
print(open(config_path).read())

"""Configuration for the probation-q-a-agent agent."""

from agents._base.config import AgentBaseConfig


class ProbationQAAgentConfig(AgentBaseConfig):
    """ProbationQAAgent agent configuration.

    Extends the base config with agent-specific settings and defaults.
    """

    agent_name: str = "probation-q-a-agent"
    agent_deployment_name: str = "gpt-5.1"
    agent_instructions_path: str = "agents/probation_q_a_agent/instructions.md"



## Step 4: Inspect Generated Instructions

View the placeholder system prompt. You'll customise this with your agent's role and capabilities.

In [9]:
instructions_path = f"../agents/{module_name}/instructions.md"
print(open(instructions_path).read())

# Probation Q A Agent Agent

You are a helpful assistant called **Probation Q A Agent**.
Your role is to assist users with their tasks.

## Capabilities

- Answer questions clearly and concisely
- Use available tools when they can help answer a question
- Provide examples when appropriate

## Guidelines

- Be concise and direct in your responses
- If you're unsure about something, say so rather than guessing
- Prioritise correctness and security in all suggestions
- Follow established conventions and patterns



## Step 5: Run Generated Tests

Run the unit tests generated for your new agent to verify the scaffolding is correct.

In [10]:
!python -m pytest ../tests/{module_name}/ -v -m "not integration"

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /home/kiran/agentframework-automation/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/kiran/agentframework-automation
configfile: pyproject.toml
plugins: anyio-4.13.0, asyncio-1.3.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 6 items / 2 deselected / 4 selected                                  

../tests/probation_q_a_agent/test_tools.py::TestGreetUser::test_greets_with_name PASSED [ 25%]
../tests/probation_q_a_agent/test_tools.py::TestGreetUser::test_greets_with_empty_name PASSED [ 50%]
../tests/probation_q_a_agent/test_tools.py::TestGreetUser::test_returns_string PASSED [ 75%]
../tests/probation_q_a_agent/test_tools.py::TestGreetUser::test_contains_agent_identifier PASSED [100%]

======================= 4 passed, 2 deselected in 0.03s ===============

## Step 6: Run Your New Agent

Assemble and run the agent locally to verify it works end-to-end.

In [11]:
from dotenv import load_dotenv
load_dotenv("../.env")

# Reset cached credential so it picks up AZURE_AUTHORITY_HOST from .env
from agents._base.client import reset_credential
reset_credential()

# Reload registry to pick up the new agent
import importlib
import agents.registry
importlib.reload(agents.registry)
from agents.registry import REGISTRY

from agents._base.agent_factory import create_agent

entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()
agent = create_agent(config)

print(f"✓ Agent '{config.agent_name}' assembled")
print(f"  Model: {config.agent_deployment_name}")


SSL verification disabled (DISABLE_SSL_VERIFY=true). Do NOT use this in production.


✓ Agent 'code-helper' assembled
  Model: gpt-5.1


In [12]:
# Send a test prompt
result = await agent.run("Hello! Please greet Alice using your greeting tool.")
response_text = result.text if hasattr(result, "text") else str(result)
print(f"Agent response: {response_text}")

Agent response: Hello, Alice! I'm the Probation Q A Agent. How can I assist you today?


## Step 7: Test as HTTP Service (Optional)

You can also test the agent as an HTTP service. Run this in a terminal:

```bash
AGENT_NAME=probation-q-a-agent python app.py
```

Then test with curl:

```bash
curl -X POST http://localhost:8088/responses \
  -H "Content-Type: application/json" \
  -d '{"input": "Hello!"}'
```

## Step 8: Delete the Agent (Optional)

Remove the scaffolded agent from the codebase — deletes the agent directory,
test directory, and registry entry.

⚠️ **This is destructive and cannot be undone.** Only run this if you want to completely remove the agent.

In [ ]:
# Uncomment to delete the scaffolded agent (this is irreversible)
#!python ../scripts/delete_agent.py --name {AGENT_NAME} --force

## Next Steps

1. Edit `agents/<your_agent>/instructions.md` with your agent's system prompt
2. Add custom tools in `agents/<your_agent>/tools/` — see the [Custom Tools Guide](../docs/custom-tools-guide.md)
3. (Optional) Set defaults in `agents/<your_agent>/config.py` (model, temperature, etc.)
4. See the [Scaffolding Guide](../docs/scaffolding-guide.md) for full reference
5. Continue to **04_deploy_to_aca.ipynb** to deploy your agent to Azure Container Apps
6. See the [Deployment Guide](../docs/deployment-guide.md) for full deployment reference